# Mk33: RAG vs KV-cache eviction (two models, confidence intervals)

Compares, at an equal memory budget, keeping tokens in the model's KV cache (random, streaming, key-norm, H2O) against cutting the text before the model reads it (truncation, and BM25 retrieval, the same RAG as Mk33's `retrieve` mode). 8 windows per model, bootstrap 95% intervals.

**Before starting:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all.

Each model takes roughly 20 to 30 minutes on a T4. Results download after **each** model, so a disconnect does not lose finished work. If the session drops, reconnect and Run all again: finished measurements are skipped.

## 1. Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name())
%cd /content
!test -d Mk33 || git clone -q https://github.com/dulguuntsatsralt-cell/Mk33
%cd /content/Mk33
!git pull -q
!pip install -q -e '.[mk33-kv,dev]'
!git log --oneline -3

## 2. Qwen2.5-3B

In [ ]:
!python scripts/mk33/memory_quality.py --model Qwen/Qwen2.5-3B-Instruct --dtype float16 \
    --out results/memq_qwen 2>&1 | grep -v -i warning | grep -v "Loading weights"

In [ ]:
!cd /content/Mk33 && zip -qr /content/memq_qwen.zip results/memq_qwen
from google.colab import files
files.download("/content/memq_qwen.zip")

## 3. Llama 3.2 3B (second model family)

In [ ]:
!python scripts/mk33/memory_quality.py --model unsloth/Llama-3.2-3B-Instruct --dtype float16 \
    --out results/memq_llama 2>&1 | grep -v -i warning | grep -v "Loading weights"

In [ ]:
!cd /content/Mk33 && zip -qr /content/memq_llama.zip results/memq_llama
from google.colab import files
files.download("/content/memq_llama.zip")

Send both zips (`memq_qwen.zip`, `memq_llama.zip`) back to Claude.